In [0]:
from pyspark.sql import functions as F

# 1) Read CSV
df_raw = (
    spark.read.option("header", "true")
    .csv("/Volumes/retail/online_data/online_data/daily_gym_attendance_workout_data.csv")   # change path if needed
)

In [0]:
display(df_raw)

In [0]:

# 2) Simple cleaning + typing (Silver)
df = (
    df_raw
    .withColumn("member_id", F.col("member_id").cast("int"))
    .withColumn("age", F.col("age").cast("int"))
    .withColumn(
        "visit_date",
        F.when(
            F.col("visit_date").rlike(r"^\d-\d-\d$"),
            F.to_date("visit_date", "dd-MM-yyyy")
        ).otherwise(
            F.to_date("visit_date", "yyyy-MM-dd")
        )
    )
    .withColumn("workout_duration_minutes", F.col("workout_duration_minutes").cast("int"))
    .withColumn("calories_burned", F.col("calories_burned").cast("int"))
    .withColumn("gender", F.initcap(F.trim("gender")))
    .withColumn("membership_type", F.initcap(F.trim("membership_type")))
    .withColumn("workout_type", F.initcap(F.trim("workout_type")))
    .withColumn("attendance_status", F.initcap(F.trim("attendance_status")))
    .withColumn("check_in_time", F.trim("check_in_time"))
    .withColumn("is_present", F.when(F.col("attendance_status") == "Present", 1).otherwise(0))
    .withColumn(
        "calories_per_min",
        F.round(
            F.col("calories_burned") /
            F.when(F.col("workout_duration_minutes") == 0, None)
            .otherwise(F.col("workout_duration_minutes")),
            2
        )
    )
)

# Optional: drop bad rows
df = df.filter(F.col("member_id").isNotNull() & F.col("visit_date").isNotNull())

# 3) Write Silver table (optional)
df.write.mode("overwrite").format("delta").saveAsTable("retail.gym_data.silver_member_visits")


In [0]:
%sql

select * from retail.gym_data.silver_member_visits

In [0]:

# 4) Gold: simple KPIs
daily_kpis = (
    df.groupBy("visit_date")
      .agg(
          F.count("*").alias("total_records"),
          F.sum("is_present").alias("present_count"),
          (F.count("*") - F.sum("is_present")).alias("absent_count"),
          F.round(F.avg("workout_duration_minutes"), 2).alias("avg_duration_min"),
          F.round(F.avg("calories_burned"), 2).alias("avg_calories"),
          F.round(F.avg("calories_per_min"), 2).alias("avg_calories_per_min"),
      )
      .withColumn("present_rate", F.round(F.col("present_count") / F.col("total_records"), 3))
)

workout_kpis = (
    df.groupBy("workout_type")
      .agg(
          F.count("*").alias("sessions"),
          F.round(F.avg("workout_duration_minutes"), 2).alias("avg_duration_min"),
          F.round(F.avg("calories_burned"), 2).alias("avg_calories"),
          F.round(F.avg("calories_per_min"), 2).alias("avg_calories_per_min"),
          F.round(F.avg("is_present"), 3).alias("present_rate"),
      )
)

# 5) Write Gold tables (optional)
daily_kpis.write.mode("overwrite").format("delta").saveAsTable("retail.gym_data.gold_daily_kpis")
workout_kpis.write.mode("overwrite").format("delta").saveAsTable("retail.gym_data.gold_workout_kpis")


display(daily_kpis.orderBy("visit_date"))
display(workout_kpis.orderBy(F.desc("sessions")))

In [0]:
from pyspark.sql import functions as F, Window as W

# 1) Monthly trend (MoM)
monthly_trend = (
    df.withColumn("visit_month", F.date_format("visit_date", "yyyy-MM"))
      .groupBy("visit_month")
      .agg(
          F.count("*").alias("total_visits"),
          F.sum("is_present").alias("present_visits"),
          F.round(F.avg("calories_burned"), 2).alias("avg_calories"),
          F.round(F.avg("workout_duration_minutes"), 2).alias("avg_duration_min"),
      )
      .withColumn("present_rate", F.round(F.col("present_visits") / F.col("total_visits"), 3))
      .orderBy("visit_month")
)

w_m = W.orderBy("visit_month")
monthly_trend = (
    monthly_trend
    .withColumn("prev_present_rate", F.lag("present_rate").over(w_m))
    .withColumn("mom_present_rate_change", F.round(F.col("present_rate") - F.col("prev_present_rate"), 3))
    .withColumn("prev_avg_calories", F.lag("avg_calories").over(w_m))
    .withColumn("mom_avg_calories_change", F.round(F.col("avg_calories") - F.col("prev_avg_calories"), 2))
)

# 2) Daily trend + rolling 3-visit average (simple rolling example)
daily_trend = (
    df.groupBy("visit_date")
      .agg(
          F.count("*").alias("visits"),
          F.sum("is_present").alias("present_visits"),
          F.round(F.avg("calories_burned"), 2).alias("avg_calories")
      )
      .withColumn("present_rate", F.round(F.col("present_visits") / F.col("visits"), 3))
      .orderBy("visit_date")
)

w_roll = W.orderBy("visit_date").rowsBetween(-2, 0)  # last 3 days/rows
daily_trend = (
    daily_trend
    .withColumn("present_rate_roll3", F.round(F.avg("present_rate").over(w_roll), 3))
    .withColumn("avg_calories_roll3", F.round(F.avg("avg_calories").over(w_roll), 2))
)

# Optional: save
monthly_trend.write.mode("overwrite").format("delta").saveAsTable("retail.gym_data.gold_monthly_trend")
daily_trend.write.mode("overwrite").format("delta").saveAsTable("retail.gym_data.gold_daily_trend")

display(monthly_trend)
display(daily_trend)
